In [0]:
import os
import json
# from openai import OpenAI
from pyspark.sql.functions import col, max
from pyspark.sql.functions import current_timestamp
from pyspark.sql.utils import AnalysisException

In [0]:
# def ai_col_classifier(table_path):
#     """
#     Classifies columns into DIM or FACT using a Databricks Model Serving endpoint
#     via the OpenAI SDK.
#     """
#     # Setup OpenAI client for databricks
#     DATABRICKS_TOKEN = dbutils.secrets.get(scope="databricks", key="api_token")
#     client = OpenAI(
#         api_key=DATABRICKS_TOKEN,
#         base_url="https://7474650070416514.ai-gateway.cloud.databricks.com/mlflow/v1"
#     )

#     # Extract metadata from information schema
#     parts = table_path.split(".")
#     catalog, schema, table = parts[0], parts[1], parts[2]

#     meta_df = spark.read.table(f"{catalog}.information_schema.columns") \
#         .where(f"table_schema = '{schema}' AND table_name = '{table}'") \
#         .select("column_name", "data_type", "comment")

#     context = "; ".join([f"{r.column_name} ({r.data_type}): {r.comment}" for r in meta_df.collect()])

#     # Call model
#     chat_completion = client.chat.completions.create(
#         messages=[
#             {
#                 "role": "system", 
#                 "content": "You are a Data Architect. Classify columns into 'DIM' (descriptive/keys) or 'FACT' (measures/metrics)."
#             },
#             {
#                 "role": "user", 
#                 "content": f"Return ONLY a JSON object: {{\"DIM\": [\"col1\"], \"FACT\": [\"col2\"]}}. Context: {context}"
#             }
#         ],
#         model="col_classifier",
#         max_tokens=1024,
#         temperature=0
#     )

#     raw_response = chat_completion.choices[0].message.content

#     # Clean and parse response
#     clean_json = raw_response.strip()
#     if clean_json.startswith("```"):
#         clean_json = clean_json.split("\n", 1)[-1].rsplit("\n", 1)[0].strip()
#     if clean_json.lower().startswith("json"):
#         clean_json = clean_json[4:].strip()

#     try:
#         return json.loads(clean_json)
#     except json.JSONDecodeError as e:
#         print(f"Failed to parse AI response for {table}: {raw_response}")
#         return {"DIM": [], "FACT": []}

In [0]:
def find_schema(source_df, target_table_path):
    """
    Loads target schema metadata and casts source_df columns to match.
    """
    # Get schema without reading data (Metadata only)
    target_schema = spark.table(target_table_path).schema

    for field in target_schema:
        if field.name in source_df.columns:
            # Explicitly cast to the target data type
            source_df = source_df.withColumn(field.name, col(field.name).cast(field.dataType))

    return source_df

In [0]:
def update_dim_fact(catalog, domain):
    # Get silver table paths
    silver_tables_df = spark.read.table(f"{catalog}.information_schema.tables") \
        .where(col("table_schema") == domain)\
        .where(col("table_name").like("silver_%"))

    rows = silver_tables_df.collect()
    if not rows:
        print(f"No silver tables found in {catalog}.{domain}")
        return

    for row in rows:
        silver_path = f"{catalog}.{domain}.{row.table_name}"
        gold_dim_path = silver_path.replace("silver", "dim")
        gold_fact_path = silver_path.replace("silver", "fact")

        print(f"Processing Silver to Gold: {silver_path}...")

        # Load silver data
        silver_df = spark.read.table(silver_path)

        dim_df = None
        fact_df = None

        try:
            # Find existing gold tables
            dim_cols = spark.table(gold_dim_path).columns
            fact_cols = spark.table(gold_fact_path).columns

            # Merge columns
            merged_dim_cols = list(set(dim_cols) | set(silver_df.columns))
            merged_fact_cols = list(set(fact_cols) | set(silver_df.columns))

            # Align types based on existing gold schema 
            dim_df = find_schema(silver_df.select(*[c for c in merged_dim_cols if c in silver_df.columns]), gold_dim_path)
            fact_df = find_schema(silver_df.select(*[c for c in merged_fact_cols if c in silver_df.columns]), gold_fact_path)

        except AnalysisException:
            print(f"Gold tables not found for {row.table_name}.")
            print(f"Using manual fact/dim file...")
            table_name = row.table_name.replace("silver_", "")

            mapping_file_path = f"./dim_fact_mapping/{table_name}.json"

            if os.path.exists(mapping_file_path):
                with open(mapping_file_path, "r") as f:
                    mapping = json.load(f)
                    dim_cols = mapping.get("dim", [])
                    fact_cols = mapping.get("fact", [])
            else:
                print(f"Please input a mapping file under ./dim_fact_mapping/{table_name}.json")
                # Infer suitable fact an dim fields based on metadata
                # print("No manual files found. Consulting AI...")
                # classification = ai_col_classifier(silver_path)

                # dim_cols = [c for c in classification.get("DIM", []) if c in silver_df.columns]
                # fact_cols = [c for c in classification.get("FACT", []) if c in silver_df.columns]

            dim_df = silver_df.select(*dim_cols)
            fact_df = silver_df.select(*fact_cols)

        # Write dimension table
        if dim_df:
            dim_df.distinct().withColumn("_load_dts", current_timestamp()) \
                .write\
                .format("delta")\
                .mode("overwrite")\
                .option("overwriteSchema", "true") \
                .saveAsTable(gold_dim_path)
            print(f"   - Successfully updated DIM: {gold_dim_path}")

        # Write fact table 
        if fact_df:
            fact_df.withColumn("_load_dts", current_timestamp()) \
                .write\
                .format("delta")\
                .mode("append")\
                .option("mergeSchema", "true") \
                .saveAsTable(gold_fact_path)
            print(f"   - Successfully updated FACT: {gold_fact_path}")

    print(f"\nCompleted all dim/fact updates for {catalog}.{domain} schema.")

In [0]:
update_dim_fact(dbutils.widgets.get("CATALOG"), dbutils.widgets.get("DOMAIN"))